# Feature Engineering

In [1]:
import warnings
warnings.filterwarnings('ignore')

import re
import math
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
import pickle

## Data Ingestion

In [2]:
dataset = load_dataset("ms_marco", "v2.1", split="validation")
marco_df = dataset.to_pandas()
marco_df.head()

,answers,passages,query,query_id,query_type,wellFormedAnswers
0,[A corporation is a company or group of people...,"{'is_selected': [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]...",. what is a corporation?,1102432,DESCRIPTION,[]
1,[Rachel Carson writes The Obligation to Endure...,"{'is_selected': [0, 0, 0, 0, 1, 1, 0, 0, 0, 0]...",why did rachel carson write an obligation to e...,1102431,DESCRIPTION,[]
2,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",why did the progressive movement fail to advan...,1102421,DESCRIPTION,[]
3,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",why do police need to understand what the fore...,1102315,DESCRIPTION,[]
4,[No Answer Present.],"{'is_selected': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...",do owls eat in the day,1101280,NUMERIC,[]


| Column | Type | Notes |
| :--- | :--- | :--- |
| query | string | The search query |
| passages | dict | Contains passage_text (list) and is_selected (list) |
| query_id | int | Unique query identifier |

In [3]:
marco_df.shape

(101093, 6)

Current size might be a bit too large for development purposes. Would be sub-sampling to 10,000 rows using random_state of 42 to ensure reproducibility

In [4]:
marco_df_samp = marco_df.sample(n=10000, random_state=42).reset_index(drop=True)

## Data Pre-processing

### Explode Rows

In [5]:
marco_df.columns

Index(['answers', 'passages', 'query', 'query_id', 'query_type',
       'wellFormedAnswers'],
      dtype='object')

In [6]:
marco_df_exp = marco_df_samp.copy()
marco_df_exp['passage_text'] = marco_df_samp['passages'].apply(lambda x: x['passage_text'])
marco_df_exp['is_selected'] = marco_df_samp['passages'].apply(lambda x: x['is_selected'])

marco_df_exp = marco_df_exp.explode(['passage_text', 'is_selected']).reset_index(drop=True)
marco_df_exp['label'] = marco_df_exp['is_selected'].astype(int)
print(marco_df_exp.shape) 

(99840, 9)


In [7]:
valid_query_ids = marco_df_exp.groupby('query_id')['is_selected'].sum()
valid_query_ids = valid_query_ids[valid_query_ids > 0].index
marco_df_exp = marco_df_exp[marco_df_exp['query_id'].isin(valid_query_ids)]

print(f"Remaining rows: {marco_df_exp.shape[0]:,}")
print(f"Unique queries: {marco_df_exp['query_id'].nunique():,}")

Remaining rows: 54,281
Unique queries: 5,435


## Feature Engineering

#### Lexical Matching Features
*"Do the query words actually appear in the passage?"*

| Feature | Why It Matters |
| :--- | :--- |
| BM25 score | The gold standard sparse retrieval signal, what Bing/Google used before neural models |
| TF-IDF cosine similarity | Measures vocabulary overlap between query and passage |
| Query term coverage | % of query words found in the passage |
| Exact match flag | Binary - does the full query appear verbatim in the passage? |

#### Semantic Features
*"Do they mean the same thing, even with different words?"*
| Feature | Why It Matters |
| :--- | :--- |
| TF-IDF vector similarity | Captures related vocabulary beyond exact matches |
| Query-passage Jaccard similarity | Simple word overlap ratio |

We may substitute TF-IDF for BERT depending on computing power

#### Statistical / Length Features
*"What does the structure of the passage tell us?"*

| Feature | Why It Matters |
| :--- | :--- |
| Passage length (word count) | Longer passages may dilute relevance |
| Query length (word count) | Longer queries are more specific |
| Length ratio | query length / passage length |
| Passage position | Earlier passages in the original list tend to be more relevant |

#### Label 

| Column | Role |
| :--- | :--- |
| is_selected | Our binary target where 1 = relevant, 0 = not relevant |




#### Query Length, Passage Length, Length Ratio

In [8]:
marco_df_exp['query_length'] = marco_df_exp['query'].str.split().str.len()
marco_df_exp['passage_length'] = marco_df_exp['passage_text'].str.split().str.len()
marco_df_exp['length_ratio'] = marco_df_exp['query_length'] / (marco_df_exp['passage_length'] + 1)
print(marco_df_exp[['query_length', 'passage_length', 'length_ratio']].describe())

       query_length  passage_length  length_ratio
count  54281.000000    54281.000000  54281.000000
mean       5.907574       51.209889      0.126205
std        2.540894       18.997756      0.074595
min        2.000000        1.000000      0.014286
25%        4.000000       40.000000      0.078431
50%        6.000000       48.000000      0.111111
75%        7.000000       57.000000      0.153846
max       30.000000      196.000000      2.500000


#### Exact match flag, Query term coverage

In [9]:
def exact_match(query, passage):
    return int(str(query).lower() in str(passage).lower())

def query_term_coverage(query, passage):
    query_words = set(str(query).lower().split())
    passage_words = set(str(passage).lower().split())
    if len(query_words) == 0:
        return 0.0
    return len(query_words & passage_words) / len(query_words)

In [10]:
marco_df_exp['exact_match'] = marco_df_exp.apply(
    lambda row: exact_match(row['query'], row['passage_text']), axis=1
)

marco_df_exp['query_term_coverage'] = marco_df_exp.apply(
    lambda row: query_term_coverage(row['query'], row['passage_text']), axis=1
)

In [11]:
print(marco_df_exp[['exact_match', 'query_term_coverage']].describe())

        exact_match  query_term_coverage
count  54281.000000         54281.000000
mean       0.010464             0.441115
std        0.101758             0.240172
min        0.000000             0.000000
25%        0.000000             0.272727
50%        0.000000             0.428571
75%        0.000000             0.600000
max        1.000000             1.000000


#### Jaccard Similarity

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

In [12]:
def jaccard_similarity(query, passage):
    query_words = set(str(query).lower().split())
    passage_words = set(str(passage).lower().split())
    if len(query_words | passage_words) == 0:
        return 0.0
    return len(query_words & passage_words) / len(query_words | passage_words)

In [13]:
marco_df_exp['jaccard_similarity'] = marco_df_exp.apply(
    lambda row: jaccard_similarity(row['query'], row['passage_text']), axis=1
)

In [14]:
print(marco_df_exp[['jaccard_similarity']].describe())

       jaccard_similarity
count        54281.000000
mean             0.063802
std              0.046048
min              0.000000
25%              0.031250
50%              0.056818
75%              0.085714
max              1.000000


#### TF-IDF Similarity

- TF-IDF Weighting
$$w_{t,d} = \text{tf}_{t,d} \times \log\left(\frac{N}{\text{df}_t}\right)$$

- Cosine Similarity (The "Similarity" Part)
$$\text{similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \sqrt{\sum_{i=1}^{n} B_i^2}}$$

In [15]:
all_text = pd.concat([marco_df_exp['query'], marco_df_exp['passage_text']], ignore_index=True)
vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True)
vectorizer.fit(all_text)

BATCH_SIZE = 5000
tfidf_scores = []

for i in tqdm(range(0, len(marco_df_exp), BATCH_SIZE), desc="TF-IDF batches"):
    batch = marco_df_exp.iloc[i:i+BATCH_SIZE]
    query_vecs = vectorizer.transform(batch['query'])
    passage_vecs = vectorizer.transform(batch['passage_text'])
    scores = cosine_similarity(query_vecs, passage_vecs).diagonal()
    tfidf_scores.extend(scores)

marco_df_exp['tfidf_cosine_sim'] = tfidf_scores


TF-IDF batches: 100%|██████████| 11/11 [00:03<00:00,  3.65it/s]


In [16]:
marco_df_exp['tfidf_cosine_sim'].describe()

count    54281.000000
mean         0.194105
std          0.139623
min          0.000000
25%          0.088628
50%          0.174015
75%          0.279113
max          1.000000
Name: tfidf_cosine_sim, dtype: float64

In [17]:
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

#### BM25

BM25 is a bag-of-words retrieval function that ranks a set of documents based on the query terms appearing in each document, regardless of their proximity within the document. It is a family of scoring functions with slightly different components and parameters. One of the most prominent instantiations of the function is as follows.

$$\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$


- $f(q_i, D)$: Number of times the query term $q_i$ occurs in document $D$
- $|D|$: Length of document $D$ (in words)
- $\text{avgdl}$: Average document length in the collection
- $k_1$: Term frequency saturation parameter  
  - Typically $k_1 \in [1.2, 2.0]$
- $b$: Length normalization parameter  
  - Commonly set to $b = 0.75$


The inverse document frequency weight for a query term \( q_i \) is usually computed as:
$$\text{IDF}(q_i) = \ln \left( \frac{N - n(q_i) + 0.5}{n(q_i) + 0.5} + 1 \right)$$

where:
- $(N)$: Total number of documents in the collection
- $(n(q_i))$: Number of documents containing term $(q_i)$

In [18]:

print("Tokenizing passages...")
tokenized_passages = [str(p).lower().split() for p in marco_df_exp['passage_text']]
bm25 = BM25Okapi(tokenized_passages)
print(f"BM25 index built over {len(tokenized_passages):,} passages")

# Get unique queries
unique_queries = marco_df_exp[['query_id', 'query']].drop_duplicates()
print(f"Scoring {len(unique_queries):,} unique queries...")

# Score each unique query once
bm25_score_map = {}
for _, row in tqdm(unique_queries.iterrows(), total=len(unique_queries), desc="BM25 scoring"):
    query_tokens = str(row['query']).lower().split()
    bm25_score_map[row['query_id']] = bm25.get_scores(query_tokens)

# Map scores back
marco_df_exp = marco_df_exp.reset_index(drop=True)
marco_df_exp['bm25_score'] = [
    bm25_score_map[row['query_id']][idx]
    for idx, row in marco_df_exp.iterrows()
]


Tokenizing passages...
BM25 index built over 54,281 passages
Scoring 5,435 unique queries...


BM25 scoring: 100%|██████████| 5435/5435 [04:16<00:00, 21.15it/s]


In [19]:
with open('bm25_scores.pkl', 'wb') as f:
    pickle.dump(marco_df_exp['bm25_score'].tolist(), f)

with open('bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)

print(f"BM25 done — saved to bm25_scores.pkl and bm25_index.pkl")
print(marco_df_exp['bm25_score'].describe().round(3))

BM25 done — saved to bm25_scores.pkl and bm25_index.pkl
count    54281.000
mean        15.491
std         10.302
min          0.000
25%          8.263
50%         14.647
75%         21.594
max        119.100
Name: bm25_score, dtype: float64


#### Passage Position

In [20]:
marco_df_exp['passage_position'] = marco_df_exp.groupby('query_id').cumcount()

### Final Check of Features

In [21]:
feature_cols = [
    'query_length', 'passage_length', 'length_ratio',
    'exact_match', 'query_term_coverage', 'jaccard_similarity',
    'tfidf_cosine_sim', 'bm25_score', 'passage_position'
]

In [22]:
print("Feature Summary:")
print(marco_df_exp[feature_cols].describe().round(3))

Feature Summary:
       query_length  passage_length  length_ratio  exact_match  \
count     54281.000       54281.000     54281.000    54281.000   
mean          5.908          51.210         0.126        0.010   
std           2.541          18.998         0.075        0.102   
min           2.000           1.000         0.014        0.000   
25%           4.000          40.000         0.078        0.000   
50%           6.000          48.000         0.111        0.000   
75%           7.000          57.000         0.154        0.000   
max          30.000         196.000         2.500        1.000   

       query_term_coverage  jaccard_similarity  tfidf_cosine_sim  bm25_score  \
count            54281.000           54281.000         54281.000   54281.000   
mean                 0.441               0.064             0.194      15.491   
std                  0.240               0.046             0.140      10.302   
min                  0.000               0.000             0.000    

In [23]:
print(f"\nShape: {marco_df_exp.shape}")


Shape: (54281, 18)


In [24]:
print(f"Missing values:\n{marco_df_exp[feature_cols].isnull().sum()}")

Missing values:
query_length           0
passage_length         0
length_ratio           0
exact_match            0
query_term_coverage    0
jaccard_similarity     0
tfidf_cosine_sim       0
bm25_score             0
passage_position       0
dtype: int64


In [25]:
# Save final dataframe
marco_df_exp.to_pickle('df_features.pkl')
print("All features saved to df_features.pkl")

All features saved to df_features.pkl


### Model Dev Next Steps - If you don't want to run feature engineering

In [26]:
'''from google.colab import drive
drive.mount('/content/drive')'''

import numpy as np 
import pickle
import pandas as pd

#Load the finished dataframe — skip all feature engineering
df_exploded = pd.read_pickle('df_features.pkl')

# Load vectorizer and BM25 index if needed later
with open('tfidf_vectorizer.pkl', 'rb') as f:
     vectorizer = pickle.load(f)

with open('bm25_index.pkl', 'rb') as f:
     bm25 = pickle.load(f)

print(df_exploded.shape)
print(df_exploded.columns.tolist())

(54281, 18)
['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers', 'passage_text', 'is_selected', 'label', 'query_length', 'passage_length', 'length_ratio', 'exact_match', 'query_term_coverage', 'jaccard_similarity', 'tfidf_cosine_sim', 'bm25_score', 'passage_position']


# Performing additional feature engineering 
optional to run, would just create new feautures in addition to the ones created above. not sure if this will impact model performance yet

### 1. additional interaction features 
### 2. semantic similarity with sentence transformers

#### handle class imbalance, save new pkl file 

### Semantic similarity with sentence transformers 

In [27]:
# Load existing features
df = pd.read_pickle('df_features.pkl')

with open('tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

### Stopword filtering (fixes Jaccard & term overlap inflation)

In [28]:
STOPWORDS = {
    'the','is','are','was','were','a','an','and','or','but','in','on',
    'at','to','for','of','with','by','from','that','this','it','as',
    'be','have','has','had','do','does','did','will','would','could',
    'should','may','might','what','which','who','how','when','where','why'
}

def clean_tokens(text: str) -> set:
    tokens = set(str(text).lower().split())
    return tokens - STOPWORDS

# Recompute Jaccard and term coverage with stopwords removed
def jaccard_clean(query, passage):
    q = clean_tokens(query)
    p = clean_tokens(passage)
    if len(q | p) == 0:
        return 0.0
    return len(q & p) / len(q | p)

def term_coverage_clean(query, passage):
    q = clean_tokens(query)
    p = clean_tokens(passage)
    if len(q) == 0:
        return 0.0
    return len(q & p) / len(q)

print("Computing stopword-filtered overlap features...")
tqdm.pandas()

df['jaccard_clean']        = df.progress_apply(
    lambda r: jaccard_clean(r['query'], r['passage_text']), axis=1)

df['term_coverage_clean']  = df.progress_apply(
    lambda r: term_coverage_clean(r['query'], r['passage_text']), axis=1)

print(df[['jaccard_clean','term_coverage_clean']].describe().round(3))

Computing stopword-filtered overlap features...


100%|██████████| 54281/54281 [00:00<00:00, 107825.53it/s]

       jaccard_clean  term_coverage_clean
count      54281.000            54281.000
mean           0.051                0.438
std            0.044                0.305
min            0.000                0.000
25%            0.026                0.250
50%            0.042                0.500
75%            0.071                0.667
max            1.000                1.000


### query type flag 

In [29]:
import re

QUESTION_WORDS = {'what','when','where','who','why','how','which','is','are','can','does','do'}

def query_type(query: str) -> int:
    """
    0 = navigational/other
    1 = question (starts with a question word)
    2 = how-to (starts with 'how')
    """
    first = str(query).lower().strip().split()[0] if query else ''
    if first == 'how':
        return 2
    elif first in QUESTION_WORDS:
        return 1
    else:
        return 0

def has_question_mark(query: str) -> int:
    return int('?' in str(query))

df['query_type']          = df['query'].apply(query_type)
df['has_question_mark']   = df['query'].apply(has_question_mark)
df['query_word_count']    = df['query'].apply(lambda q: len(str(q).split()))

print("Query type distribution:")
print(df['query_type'].value_counts())
print("\nQuestion mark rate:", df['has_question_mark'].mean().round(3))

Query type distribution:
query_type
1    31672
0    15955
2     6654
Name: count, dtype: int64

Question mark rate: 0.085


### Softened phrase match (replaces near-constant exact_match)


In [30]:
def ngram_overlap(query: str, passage: str, n: int = 3) -> float:
    """
    Fraction of query n-grams that appear in the passage.
    Softer signal than exact full-query match.
    """
    q_tokens = str(query).lower().split()
    p_text   = str(passage).lower()
    
    if len(q_tokens) < n:
        # fall back to unigram coverage for short queries
        return term_coverage_clean(query, passage)
    
    q_ngrams = [' '.join(q_tokens[i:i+n]) for i in range(len(q_tokens) - n + 1)]
    matches  = sum(1 for ng in q_ngrams if ng in p_text)
    return matches / len(q_ngrams) if q_ngrams else 0.0

print("Computing 3-gram overlap (soft phrase match)...")
df['trigram_overlap'] = df.progress_apply(
    lambda r: ngram_overlap(r['query'], r['passage_text'], n=3), axis=1)

print(df[['exact_match','trigram_overlap']].describe().round(3))
print(f"\nTrigram overlap fires {(df['trigram_overlap'] > 0).mean():.1%} of the time vs "
      f"exact match {df['exact_match'].mean():.1%}")

Computing 3-gram overlap (soft phrase match)...


100%|██████████| 54281/54281 [00:00<00:00, 157290.67it/s]

       exact_match  trigram_overlap
count    54281.000        54281.000
mean         0.010            0.056
std          0.102            0.167
min          0.000            0.000
25%          0.000            0.000
50%          0.000            0.000
75%          0.000            0.000
max          1.000            1.000

Trigram overlap fires 14.3% of the time vs exact match 1.0%


### Interaction features 

In [31]:
# Explicit interaction terms — gives DeepFM a head start on strong signal combinations
df['bm25_x_tfidf']          = df['bm25_score'] * df['tfidf_cosine_sim']
df['bm25_x_term_coverage']  = df['bm25_score'] * df['term_coverage_clean']
df['tfidf_x_jaccard']       = df['tfidf_cosine_sim'] * df['jaccard_clean']

# Normalized overlap — penalizes long passages that match by chance
df['overlap_per_passage_len'] = df['term_coverage_clean'] / (df['passage_length'] + 1)

print("Interaction features added:")
print(df[['bm25_x_tfidf','bm25_x_term_coverage',
          'tfidf_x_jaccard','overlap_per_passage_len']].describe().round(3))

Interaction features added:
       bm25_x_tfidf  bm25_x_term_coverage  tfidf_x_jaccard  \
count     54281.000             54281.000        54281.000   
mean          3.994                 8.919            0.014   
std           4.935                 9.058            0.022   
min           0.000                 0.000            0.000   
25%           0.711                 2.162            0.002   
50%           2.474                 6.310            0.007   
75%           5.492                13.466            0.018   
max         104.594               119.100            1.000   

       overlap_per_passage_len  
count                54281.000  
mean                     0.009  
std                      0.008  
min                      0.000  
25%                      0.004  
50%                      0.008  
75%                      0.013  
max                      0.100  


### Handle class imbalance and save file 

In [33]:
print("Class distribution:")
print(df['is_selected'].value_counts())
print(f"Positive rate: {df['is_selected'].mean():.3%}")

# Compute pos_weight for use in loss function during training
n_neg     = (df['is_selected'] == 0).sum()
n_pos     = (df['is_selected'] == 1).sum()
pos_weight = n_neg / n_pos
print(f"\nRecommended pos_weight for BCEWithLogitsLoss: {pos_weight:.1f}")
# Pass this into your DeepFM training:
# criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))

# Save enriched dataframe
df.to_pickle('df_features_v2.pkl')
print(f"\nSaved enriched features: {df.shape}")
print(f"New columns added: {[c for c in df.columns if c not in ['query','passage_text','is_selected','query_id']]}")

Class distribution:
is_selected
0    48482
1     5799
Name: count, dtype: int64
Positive rate: 10.683%

Recommended pos_weight for BCEWithLogitsLoss: 8.4

Saved enriched features: (54281, 27)
New columns added: ['answers', 'passages', 'query_type', 'wellFormedAnswers', 'label', 'query_length', 'passage_length', 'length_ratio', 'exact_match', 'query_term_coverage', 'jaccard_similarity', 'tfidf_cosine_sim', 'bm25_score', 'passage_position', 'jaccard_clean', 'term_coverage_clean', 'has_question_mark', 'query_word_count', 'trigram_overlap', 'bm25_x_tfidf', 'bm25_x_term_coverage', 'tfidf_x_jaccard', 'overlap_per_passage_len']
